In [33]:
"""
stock_assistant_chatbot.py

Chat assistant that:
- Accepts a NYSE ticker from the user,
- Fetches current price & 1-month history using yfinance,
- Uses OpenAI LLM to answer questions / describe the stock,
- Presents a chat interface wcontributions
ith Gradio.

Requirements:
    pip install gradio yfinance matplotlib pandas pillow openai
Before running, set your OpenAI key:
    export OPENAI_API_KEY="sk-..."
"""

import datetime
from pickle import TRUE
import yfinance as yf
import pandas as pd
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import requests
import json

load_dotenv(override=True)
client = OpenAI()  # reads API key from OPENAI_API_KEY



def search_company_ticker(company_name):
    """
    Search for ticker symbol using Yahoo Finance API
    """
    try:
        url = "https://query2.finance.yahoo.com/v1/finance/search"
        params = {
            "q": company_name,
            "quotes_count": 5,  # Get top 5 results
            "country": "United States"
        }
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if "quotes" in data and len(data["quotes"]) > 0:
            # Return the first result (most relevant)
            return data["quotes"][0]["symbol"]
        else:
            return None
            
    except Exception as e:
        print(f"Error searching for ticker: {e}")
        return None

def get_ticker_from_input(user_input):
    """
    Extract ticker from user input. If it looks like a company name, 
    try to find the corresponding ticker using Yahoo Finance API.
    """
    # Clean and normalize input
    input_clean = user_input.strip()
    
    # If it's already a ticker-like format (3-5 uppercase letters), return as is
    if len(input_clean) <= 5 and input_clean.isalpha() and input_clean.isupper():
        return input_clean
    
    # Try Yahoo Finance API search
    ticker = search_company_ticker(input_clean)
    
    if ticker:
        return ticker
    else:
        # If no match found, return None to indicate invalid company/ticker
        return None

def get_stock_data(ticker):
    """Fetch current price, previous close, and 1-month history."""
    tk = yf.Ticker(ticker)
    info = tk.info
    price = info.get("regularMarketPrice")
    prev_close = info.get("previousClose")
    hist = tk.history(period="1mo", interval="1d", actions=False)
    return price, prev_close, hist, info



def ask_llm_about_stock(ticker, price, prev_close, hist, info, user_question):
    """
    Feed the stock data + the user's question to the LLM to generate a reply.
    """
    closes = hist["Close"].tolist() if not hist.empty else []
    dates = [str(d.date()) for d in hist.index] if not hist.empty else []
    base_info = f"Ticker: {ticker}, Current price: {price} USD, Previous close: {prev_close} USD."
    prompt = f"""
You are a helpful financial assistant.
You have the following data for the NYSE stock {ticker}:
{base_info}
Dates (last month): {dates}
Closing prices: {closes}

User question: {user_question}

Give a concise, informative answer based on the data.
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

def respond(user_message, history):
    """
    Called every time the user sends a message.
    Handles both: 1) ticker only (show price), 2) ticker + question (use LLM), 3) follow-up questions (use LLM)
    """
    parts = user_message.strip().split(maxsplit=1)
    
    # Check if first part looks like a ticker/company
    first_part = parts[0]
    is_ticker_format = ((len(first_part) <= 5 and first_part.isalpha() and first_part.isupper()) or 
                       first_part.lower() in ["apple", "microsoft", "tesla", "google", "amazon", "meta", "netflix", "nvidia", "intel", "cisco", "oracle", "salesforce", "adobe"])
    
    if is_ticker_format:
        # First part is a ticker/company
        ticker_input = first_part
        
        if len(parts) == 1:
            # Single ticker/company - show current price only
            ticker = get_ticker_from_input(ticker_input)
            
            # Check if ticker lookup failed
            if ticker is None:
                reply = f"Company name not found: '{ticker_input}'. Please try a different company name or ticker symbol."
                return history + [[user_message, reply]], ""

            try:
                price, prev_close, hist, info = get_stock_data(ticker)
                if price is None:
                    reply = f"No market data found for ticker {ticker}."
                    return history + [[user_message, reply]], ""

                # Simple current price response
                now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                reply = f"**{ticker}** - Current Price: ${price:.2f}\nPrevious Close: ${prev_close:.2f}\n\n_Fetched at {now}_"

                # Return chat history with ticker symbol, and ticker for the input box
                return history + [[ticker, reply]], f"{ticker} "
            except Exception as e:
                return history + [[user_message, f"Error: {e}"]], ""
        
        else:
            # Ticker + question - use LLM
            question = parts[1]
            ticker = get_ticker_from_input(ticker_input)
            
            # Check if ticker lookup failed
            if ticker is None:
                reply = f"Company name not found: '{ticker_input}'. Please try a different company name or ticker symbol."
                return history + [[user_message, reply]], ""

            try:
                price, prev_close, hist, info = get_stock_data(ticker)
                if price is None:
                    reply = f"No market data found for ticker {ticker}."
                    return history + [[user_message, reply]], ""

                # Use LLM for the question
                answer = ask_llm_about_stock(ticker, price, prev_close, hist, info, question)
                now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                reply = f"**{ticker}**\n\n{answer}\n\n_Fetched at {now}_"

                return history + [[user_message, reply]], f"{ticker} "
            except Exception as e:
                return history + [[user_message, f"Error: {e}"]], ""
    
    else:
        # Not a ticker format - treat as follow-up question
        # Get the last ticker from chat history
        last_ticker = None
        for i in range(len(history) - 1, -1, -1):
            user_msg, bot_msg = history[i]
            if "**" in bot_msg:
                try:
                    ticker_start = bot_msg.find("**") + 2
                    ticker_end = bot_msg.find("**", ticker_start)
                    if ticker_end > ticker_start:
                        last_ticker = bot_msg[ticker_start:ticker_end]
                        break
                except:
                    pass
        
        if last_ticker is None:
            reply = "Please enter a ticker symbol or company name first (e.g., 'Apple' or 'MSFT')."
            return history + [[user_message, reply]], ""
        
        try:
            # Get fresh stock data for the conversation
            price, prev_close, hist, info = get_stock_data(last_ticker)
            if price is None:
                reply = f"No market data found for ticker {last_ticker}."
                return history + [[user_message, reply]], ""

            # Use LLM for the conversation
            answer = ask_llm_about_stock(last_ticker, price, prev_close, hist, info, user_message)
            now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            reply = f"**{last_ticker}**\n\n{answer}\n\n_Fetched at {now}_"

            return history + [[user_message, reply]], f"{last_ticker} "
        except Exception as e:
            return history + [[user_message, f"Error: {e}"]], ""

# Build Gradio UI
with gr.Blocks(title="Stock Genius") as demo:
    gr.Markdown("## 💬 Stock Genius Assistant Chatbot\n"
                "**Enter a ticker or company name** to see current price (e.g. `Apple`, `MSFT`)\n"
                "**Then ask questions** about that stock (e.g. `What's the trend?`, `Show me the volume`)")
    chatbot = gr.Chatbot(label="Conversation", height=400)
    with gr.Row():
        msg = gr.Textbox(placeholder="Enter TICKER or 'TICKER question'", label="Your message")
        submit = gr.Button("Send")


    def user_submit(user_message, chat_history):
        return respond(user_message, chat_history)

    submit.click(user_submit, [msg, chatbot], [chatbot, msg])
    msg.submit(user_submit, [msg, chatbot], [chatbot, msg])

demo.launch(share=TRUE)


/var/folders/bc/0jnxy9wn0_g12kj4w3wvzpkr0000gn/T/ipykernel_27092/2033491000.py:223: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation", height=400)


* Running on local URL:  http://127.0.0.1:7935
* Running on public URL: https://c60283b44f2696f26d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
